In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class Emergency Acuity Triage: Architecture Comparison (`models/train_oof_logistic_regression_stacking_new.ipynb`)

This notebook benchmarks and compares two hierarchical multi-class stacking architectures across all **5 triage levels (`ESI 1..5`)** on the **5-variable Emergency Department dataset (`datasets/5v_cleandf.RData`)**:

### 🏗️ Architecture Comparison Overview

#### **Pipeline 1: Branching Tree Stacking (Available Pipeline)**
- **Layer 1**: Classify `ESI 1` vs `(ESI 2, 3, 4, 5)` (1:1 RUS).
- **Layer 2**: Classify `(ESI 2, 3)` vs `(ESI 4, 5)` on non-ESI 1 cohort (1:1 RUS).
- **Layer 3A**: Classify `ESI 2` vs `ESI 3` on urgent/emergent cohort (1:1 RUS).
- **Layer 3B**: Classify `ESI 4` vs `ESI 5` on lower acuity cohort (1:1 RUS).
- **Meta-Learner**: Multinomial Logistic Regression calibrated on hierarchical tree probabilities.

#### **Pipeline 2: Sequential Acuity Peel-off Cascade (Proposed Pipeline)**
- **Layer 1 (Resuscitation Peel)**: Classify `ESI 1` vs `(ESI 2, 3, 4, 5)` (1:1 RUS).
- **Layer 2 (Non-urgent Peel)**: Classify `ESI 5` vs `(ESI 2, 3, 4)` excluding ESI 1 rows (1:1 RUS).
- **Layer 3 (Semi-urgent Peel)**: Classify `ESI 4` vs `(ESI 2, 3)` excluding ESI 1 & 5 rows (1:1 RUS).
- **Layer 4 (Emergent vs Urgent Resolution)**: Classify `ESI 2` vs `ESI 3` excluding ESI 1, 4 & 5 rows (1:1 RUS).
- **Meta-Learner**: Multinomial Logistic Regression calibrated on sequential cascade probabilities.

```mermaid
flowchart TD
    Data["Clean Dataset (5v_cleandf.RData)"] --> Split["3-Way Stratified Split: Train (98%), Val (1%), Test (1%)"]
    
    subgraph Pipe1 ["Pipeline 1: Branching Tree Stacking"]
        Split --> P1L1["Layer 1: ESI 1 vs 2,3,4,5 (1:1 RUS)"]
        P1L1 --> P1L2["Layer 2: ESI 2,3 vs 4,5 (1:1 RUS, Excl ESI 1)"]
        P1L2 --> P1L3A["Layer 3A: ESI 2 vs 3 (1:1 RUS)"]
        P1L2 --> P1L3B["Layer 3B: ESI 4 vs 5 (1:1 RUS)"]
        P1L3A & P1L3B --> P1Meta["Logistic Regression Meta-Learner (Tree Probabilities)"]
    end
    
    subgraph Pipe2 ["Pipeline 2: Sequential Acuity Peel-off Cascade"]
        Split --> P2L1["Layer 1: ESI 1 vs 2,3,4,5 (1:1 RUS)"]
        P2L1 --> P2L2["Layer 2: ESI 5 vs 2,3,4 (1:1 RUS, Excl ESI 1)"]
        P2L2 --> P2L3["Layer 3: ESI 4 vs 2,3 (1:1 RUS, Excl ESI 1 & 5)"]
        P2L3 --> P2L4["Layer 4: ESI 2 vs 3 (1:1 RUS, Excl ESI 1, 4 & 5)"]
        P2L4 --> P2Meta["Logistic Regression Meta-Learner (Cascade Probabilities)"]
    end
    
    P1Meta & P2Meta --> Benchmark["Holdout Test Benchmark: Per-Class Scoring, Confusion Matrices & ROC Curves"]
```

### 📋 Benchmark Evaluation Objectives
1. **Layer-by-Layer Sub-model Performance**: Standalone binary evaluation of all decision nodes.
2. **Raw Multi-Class Probability Chain (without LogReg)**: Argmax decisions directly from probability formulas.
3. **Calibrated Multi-Class Stacking (with LogReg)**: Calibrated 5-class triage predictions using `get_per_class_breakdown()`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & Stratified 3-Way Split (Train/Val/Test)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")
if (target_col_name %in% names(raw_df)) {
  cat("Initial ESI Target Distribution (including NAs):\n")
  print(table(raw_df[[target_col_name]], useNA = "ifany"))
  cat("------------------------------------------------------------------------\n")
}

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi <- as.character(raw_df[[target_col_name]])

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across all 15 raw features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Cleaned ESI Distribution (100% complete cases):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning based on triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (e.g., 1%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (e.g., 1% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:15], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:15],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:15],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Matrix Construction & Preprocessing
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from rpy2.robjects import r
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :15]
y_train     = train_mat_in[:, 15].astype(int)

raw_mat_val = val_mat_in[:, :15]
y_val       = val_mat_in[:, 15].astype(int)

raw_mat_ts  = test_mat_in[:, :15]
y_test      = test_mat_in[:, 15].astype(int)

def build_38_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 38), dtype=np.float64)
    X[:, :15] = raw_mat
    
    t_hr = raw_mat[:, 3]; t_sbp = raw_mat[:, 4]; t_rr = raw_mat[:, 5]; t_o2 = raw_mat[:, 6]
    pulse_min = raw_mat[:, 7]; resp_min = raw_mat[:, 8]; spo2_min = raw_mat[:, 9]; sbp_min = raw_mat[:, 10]
    pulse_max = raw_mat[:, 11]; resp_max = raw_mat[:, 12]; spo2_max = raw_mat[:, 13]; sbp_max = raw_mat[:, 14]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    X[:, 15] = (t_o2 < 90).astype(float)
    X[:, 16] = ((t_o2 > 90) & (t_o2 < 94)).astype(float)
    X[:, 17] = (t_rr < 10).astype(float)
    X[:, 18] = (t_rr > 30).astype(float)
    X[:, 19] = (t_sbp <= 90).astype(float)
    X[:, 20] = (t_sbp > 220).astype(float)
    X[:, 21] = (t_hr < 40).astype(float)
    X[:, 22] = ((t_hr > 40) & (t_hr < 60)).astype(float)
    X[:, 23] = (t_hr > 150).astype(float)
    X[:, 24] = ((t_hr > 100) & (t_hr < 150)).astype(float)
    X[:, 25] = hr_rng; X[:, 26] = rr_rng; X[:, 27] = spo2_rng; X[:, 28] = sbp_rng
    X[:, 29] = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 30] = t_hr - hr_rng
    X[:, 31] = t_sbp - sbp_rng
    X[:, 32] = t_rr - rr_rng
    X[:, 33] = t_o2 - spo2_rng
    X[:, 34] = t_o2 / np.where(t_rr == 0, 1.0, t_rr)
    X[:, 35] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max)
    X[:, 36] = hr_rng / (t_hr + 1.0)
    X[:, 37] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0
    return X

X_train_raw = build_38_feature_matrix(raw_mat_tr)
X_val_raw   = build_38_feature_matrix(raw_mat_val)
X_test_raw  = build_38_feature_matrix(raw_mat_ts)

cont_cols = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()
X_train[:, cont_cols] = scaler.fit_transform(X_train_raw[:, cont_cols])
X_val[:, cont_cols]   = scaler.transform(X_val_raw[:, cont_cols])
X_test[:, cont_cols]  = scaler.transform(X_test_raw[:, cont_cols])

print(f"Feature Matrices Ready: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Train Pipeline 1 — Branching Tree Stacking (Available Pipeline)
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING PIPELINE 1: BRANCHING TREE HIERARCHICAL STACKING")
print("=" * 80)

def random_undersample_binary(X, y_bin, ratio=1.0, seed=42):
    np.random.seed(seed)
    pos_idx = np.where(y_bin == 1)[0]
    neg_idx = np.where(y_bin == 0)[0]
    n_pos = len(pos_idx)
    n_neg_keep = min(int(n_pos * ratio), len(neg_idx))
    kept_neg_idx = np.random.choice(neg_idx, size=n_neg_keep, replace=False)
    chosen_idx = np.sort(np.concatenate([pos_idx, kept_neg_idx]))
    return X[chosen_idx], y_bin[chosen_idx]

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}

t0 = time.time()

# P1 - Layer 1: ESI 1 vs (ESI 2..5) [1:1 RUS]
X_p1_l1, y_p1_l1 = random_undersample_binary(X_train, (y_train == 1).astype(int), ratio=1.0)
p1_l1 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p1_l1.fit(X_p1_l1, y_p1_l1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# P1 - Layer 2: ESI 2,3 vs ESI 4,5 on non-ESI 1 [1:1 RUS]
m_non1_tr  = (y_train != 1); m_non1_val = (y_val != 1)
X_p1_l2, y_p1_l2 = random_undersample_binary(X_train[m_non1_tr], np.isin(y_train[m_non1_tr], [2, 3]).astype(int), ratio=1.0)
p1_l2 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p1_l2.fit(X_p1_l2, y_p1_l2, eval_set=[(X_val[m_non1_val], np.isin(y_val[m_non1_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# P1 - Layer 3A: ESI 2 vs ESI 3 on (ESI 2, 3) [1:1 RUS]
m_23_tr  = np.isin(y_train, [2, 3]); m_23_val = np.isin(y_val, [2, 3])
X_p1_l3a, y_p1_l3a = random_undersample_binary(X_train[m_23_tr], (y_train[m_23_tr] == 2).astype(int), ratio=1.0)
p1_l3a = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p1_l3a.fit(X_p1_l3a, y_p1_l3a, eval_set=[(X_val[m_23_val], (y_val[m_23_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# P1 - Layer 3B: ESI 4 vs ESI 5 on (ESI 4, 5) [1:1 RUS]
m_45_tr  = np.isin(y_train, [4, 5]); m_45_val = np.isin(y_val, [4, 5])
X_p1_l3b, y_p1_l3b = random_undersample_binary(X_train[m_45_tr], (y_train[m_45_tr] == 4).astype(int), ratio=1.0)
p1_l3b = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p1_l3b.fit(X_p1_l3b, y_p1_l3b, eval_set=[(X_val[m_45_val], (y_val[m_45_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

print(f"✓ Pipeline 1 sub-models trained in {time.time()-t0:.1f}s!")

def compute_pipeline1_probs(l1, l2, l3a, l3b, X_in):
    p1  = l1.predict_proba(X_in)[:, 1]
    p2  = l2.predict_proba(X_in)[:, 1]
    p3a = l3a.predict_proba(X_in)[:, 1]
    p3b = l3b.predict_proba(X_in)[:, 1]
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1
    P[:, 1] = (1 - p1) * p2 * p3a
    P[:, 2] = (1 - p1) * p2 * (1 - p3a)
    P[:, 3] = (1 - p1) * (1 - p2) * p3b
    P[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    return P

val_probs_p1  = compute_pipeline1_probs(p1_l1, p1_l2, p1_l3a, p1_l3b, X_val)
test_probs_p1 = compute_pipeline1_probs(p1_l1, p1_l2, p1_l3a, p1_l3b, X_test)

meta_p1 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_p1.fit(val_probs_p1, y_val)

preds_p1 = meta_p1.predict(test_probs_p1)
probs_p1 = meta_p1.predict_proba(test_probs_p1)
print("✓ Pipeline 1 (Branching Tree Stacking) Meta-Learner calibrated!")

In [ ]:
# ---------------------------------------------------------
# Step 4: Train Pipeline 2 — Sequential Acuity Peel-off Cascade (Proposed Pipeline)
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING PIPELINE 2: SEQUENTIAL ACUITY PEEL-OFF CASCADE (PROPOSED)")
print("=" * 80)

t0 = time.time()

# P2 - Layer 1: ESI 1 vs (ESI 2, 3, 4, 5) [1:1 RUS]
X_p2_l1, y_p2_l1 = random_undersample_binary(X_train, (y_train == 1).astype(int), ratio=1.0)
p2_l1 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p2_l1.fit(X_p2_l1, y_p2_l1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# P2 - Layer 2: ESI 5 vs (ESI 2, 3, 4) excluding ESI 1 [1:1 RUS]
m_p2_l2_tr  = (y_train != 1); m_p2_l2_val = (y_val != 1)
X_p2_l2, y_p2_l2 = random_undersample_binary(X_train[m_p2_l2_tr], (y_train[m_p2_l2_tr] == 5).astype(int), ratio=1.0)
p2_l2 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p2_l2.fit(X_p2_l2, y_p2_l2, eval_set=[(X_val[m_p2_l2_val], (y_val[m_p2_l2_val] == 5).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# P2 - Layer 3: ESI 4 vs (ESI 2, 3) excluding ESI 1 & 5 [1:1 RUS]
m_p2_l3_tr  = np.isin(y_train, [2, 3, 4]); m_p2_l3_val = np.isin(y_val, [2, 3, 4])
X_p2_l3, y_p2_l3 = random_undersample_binary(X_train[m_p2_l3_tr], (y_train[m_p2_l3_tr] == 4).astype(int), ratio=1.0)
p2_l3 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p2_l3.fit(X_p2_l3, y_p2_l3, eval_set=[(X_val[m_p2_l3_val], (y_val[m_p2_l3_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# P2 - Layer 4: ESI 2 vs ESI 3 excluding ESI 1, 4, 5 [1:1 RUS]
m_p2_l4_tr  = np.isin(y_train, [2, 3]); m_p2_l4_val = np.isin(y_val, [2, 3])
X_p2_l4, y_p2_l4 = random_undersample_binary(X_train[m_p2_l4_tr], (y_train[m_p2_l4_tr] == 2).astype(int), ratio=1.0)
p2_l4 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
p2_l4.fit(X_p2_l4, y_p2_l4, eval_set=[(X_val[m_p2_l4_val], (y_val[m_p2_l4_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

print(f"✓ Pipeline 2 sub-models trained in {time.time()-t0:.1f}s!")

def compute_pipeline2_probs(l1, l2, l3, l4, X_in):
    p1 = l1.predict_proba(X_in)[:, 1]  # P(ESI 1)
    p2 = l2.predict_proba(X_in)[:, 1]  # P(ESI 5 | non-1)
    p3 = l3.predict_proba(X_in)[:, 1]  # P(ESI 4 | 2,3,4)
    p4 = l4.predict_proba(X_in)[:, 1]  # P(ESI 2 | 2,3)
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1                                          # ESI 1
    P[:, 1] = (1 - p1) * (1 - p2) * (1 - p3) * p4         # ESI 2
    P[:, 2] = (1 - p1) * (1 - p2) * (1 - p3) * (1 - p4)   # ESI 3
    P[:, 3] = (1 - p1) * (1 - p2) * p3                    # ESI 4
    P[:, 4] = (1 - p1) * p2                               # ESI 5
    return P

val_probs_p2  = compute_pipeline2_probs(p2_l1, p2_l2, p2_l3, p2_l4, X_val)
test_probs_p2 = compute_pipeline2_probs(p2_l1, p2_l2, p2_l3, p2_l4, X_test)

meta_p2 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_p2.fit(val_probs_p2, y_val)

preds_p2 = meta_p2.predict(test_probs_p2)
probs_p2 = meta_p2.predict_proba(test_probs_p2)
print("✓ Pipeline 2 (Sequential Peel-Off Cascade) Meta-Learner calibrated!")

In [ ]:
# ---------------------------------------------------------
# Step 5: Holdout Test Set Benchmark & Comparative Evaluation
# ---------------------------------------------------------
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try:
            auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception:
            auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

def evaluate_binary_submodel(y_true, y_pred, y_prob, model_name, pos_label_name="Positive"):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    bal_acc = (sens + spec) / 2.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * (prec * sens) / (prec + sens) if (prec + sens) > 0 else 0.0
    try: auc_val = roc_auc_score(y_true, y_prob)
    except Exception: auc_val = 0.0
    try: pr_auc = average_precision_score(y_true, y_prob)
    except Exception: pr_auc = 0.0
    
    return {
        'Model_Node': model_name,
        f'Recall_Sens_{pos_label_name}': round(sens, 4),
        'Specificity_Neg': round(spec, 4),
        'Balanced_Accuracy': round(bal_acc, 4),
        'Precision_PPV': round(prec, 4),
        'F1_Score': round(f1, 4),
        'ROC_AUC': round(auc_val, 4),
        'PR_AUC': round(pr_auc, 4),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)
    }

# ---------------------------------------------------------
# Part A: STANDALONE SUB-MODEL BINARY NODE EVALUATION (p >= 0.50)
# ---------------------------------------------------------
# Pipeline 1 Sub-models
prob_p1_l1 = p1_l1.predict_proba(X_test)[:, 1]
m_ts_non1  = (y_test != 1)
prob_p1_l2 = p1_l2.predict_proba(X_test[m_ts_non1])[:, 1]
m_ts_23    = np.isin(y_test, [2, 3])
prob_p1_l3a = p1_l3a.predict_proba(X_test[m_ts_23])[:, 1]
m_ts_45    = np.isin(y_test, [4, 5])
prob_p1_l3b = p1_l3b.predict_proba(X_test[m_ts_45])[:, 1]

# Pipeline 2 Sub-models
prob_p2_l1 = p2_l1.predict_proba(X_test)[:, 1]
prob_p2_l2 = p2_l2.predict_proba(X_test[m_ts_non1])[:, 1]
m_ts_234   = np.isin(y_test, [2, 3, 4])
prob_p2_l3 = p2_l3.predict_proba(X_test[m_ts_234])[:, 1]
prob_p2_l4 = p2_l4.predict_proba(X_test[m_ts_23])[:, 1]

node_evals = [
    # Pipeline 1 Nodes
    evaluate_binary_submodel((y_test == 1).astype(int), (prob_p1_l1 >= 0.5).astype(int), prob_p1_l1, 'P1 - L1: ESI 1 vs 2,3,4,5', pos_label_name='ESI1'),
    evaluate_binary_submodel(np.isin(y_test[m_ts_non1], [2, 3]).astype(int), (prob_p1_l2 >= 0.5).astype(int), prob_p1_l2, 'P1 - L2: ESI 2,3 vs 4,5 (Excl ESI 1)', pos_label_name='ESI23'),
    evaluate_binary_submodel((y_test[m_ts_23] == 2).astype(int), (prob_p1_l3a >= 0.5).astype(int), prob_p1_l3a, 'P1 - L3A: ESI 2 vs 3 (Excl ESI 1,4,5)', pos_label_name='ESI2'),
    evaluate_binary_submodel((y_test[m_ts_45] == 4).astype(int), (prob_p1_l3b >= 0.5).astype(int), prob_p1_l3b, 'P1 - L3B: ESI 4 vs 5 (Excl ESI 1,2,3)', pos_label_name='ESI4'),
    
    # Pipeline 2 Nodes
    evaluate_binary_submodel((y_test == 1).astype(int), (prob_p2_l1 >= 0.5).astype(int), prob_p2_l1, 'P2 - L1: ESI 1 vs 2,3,4,5', pos_label_name='ESI1'),
    evaluate_binary_submodel((y_test[m_ts_non1] == 5).astype(int), (prob_p2_l2 >= 0.5).astype(int), prob_p2_l2, 'P2 - L2: ESI 5 vs 2,3,4 (Excl ESI 1)', pos_label_name='ESI5'),
    evaluate_binary_submodel((y_test[m_ts_234] == 4).astype(int), (prob_p2_l3 >= 0.5).astype(int), prob_p2_l3, 'P2 - L3: ESI 4 vs 2,3 (Excl ESI 1 & 5)', pos_label_name='ESI4'),
    evaluate_binary_submodel((y_test[m_ts_23] == 2).astype(int), (prob_p2_l4 >= 0.5).astype(int), prob_p2_l4, 'P2 - L4: ESI 2 vs 3 (Excl ESI 1,4,5)', pos_label_name='ESI2')
]
node_eval_df = pd.DataFrame(node_evals)

print("=" * 115)
print("   STANDALONE SUB-MODEL BINARY NODE EVALUATION (p >= 0.50)")
print("=" * 115)
print(node_eval_df[['Model_Node', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC', 'PR_AUC', 'TP', 'FP', 'TN', 'FN']].to_string(index=False))
print("=" * 115 + chr(10))

# ---------------------------------------------------------
# Part B: RAW HIERARCHICAL PROBABILITY CHAIN (WITHOUT LOGISTIC REGRESSOR)
# ---------------------------------------------------------
preds_raw_p1 = np.argmax(test_probs_p1, axis=1) + 1
preds_raw_p2 = np.argmax(test_probs_p2, axis=1) + 1

report_raw_p1 = get_per_class_breakdown(y_test, preds_raw_p1, test_probs_p1, 'Pipeline 1: Branching Tree (No LogReg)')
report_raw_p2 = get_per_class_breakdown(y_test, preds_raw_p2, test_probs_p2, 'Pipeline 2: Sequential Peel-Off (No LogReg)')

print("=" * 95)
print("   RAW PROBABILITY CHAIN (WITHOUT LOGISTIC REGRESSOR) — PIPELINE 1: BRANCHING TREE")
print("=" * 95)
print(report_raw_p1.to_string(index=False))
print("=" * 95 + chr(10))

print("=" * 95)
print("   RAW PROBABILITY CHAIN (WITHOUT LOGISTIC REGRESSOR) — PIPELINE 2: SEQUENTIAL CASCADE")
print("=" * 95)
print(report_raw_p2.to_string(index=False))
print("=" * 95 + chr(10))

# ---------------------------------------------------------
# Part C: FINAL CALIBRATED STACKING PIPELINE (WITH LOGISTIC REGRESSOR)
# ---------------------------------------------------------
report_p1 = get_per_class_breakdown(y_test, preds_p1, probs_p1, 'Pipeline 1: Branching Tree (With LogReg)')
report_p2 = get_per_class_breakdown(y_test, preds_p2, probs_p2, 'Pipeline 2: Sequential Peel-Off (With LogReg)')

print("=" * 95)
print("   FINAL STACKING PIPELINE (WITH LOGISTIC REGRESSOR) — PIPELINE 1: BRANCHING TREE")
print("=" * 95)
print(report_p1.to_string(index=False))
print("=" * 95 + chr(10))

print("=" * 95)
print("   FINAL STACKING PIPELINE (WITH LOGISTIC REGRESSOR) — PIPELINE 2: SEQUENTIAL CASCADE")
print("=" * 95)
print(report_p2.to_string(index=False))
print("=" * 95 + chr(10))

# ---------------------------------------------------------
# Part D: CONSOLIDATED ARCHITECTURE COMPARISON TABLE
# ---------------------------------------------------------
comp_rows = []
for i in range(len(report_p1)):
    cls_label = report_p1.loc[i, 'Class']
    r_p1, r_p2 = report_p1.loc[i, 'Recall'], report_p2.loc[i, 'Recall']
    s_p1, s_p2 = report_p1.loc[i, 'Specificity'], report_p2.loc[i, 'Specificity']
    b_p1, b_p2 = report_p1.loc[i, 'Balanced_Accuracy'], report_p2.loc[i, 'Balanced_Accuracy']
    a_p1, a_p2 = report_p1.loc[i, 'ROC_AUC'], report_p2.loc[i, 'ROC_AUC']
    
    comp_rows.append({
        'Class': cls_label,
        'P1_Branch_Recall': f"{r_p1*100:.2f}%",
        'P2_Seq_Recall': f"{r_p2*100:.2f}%",
        'Delta_Recall': f"{(r_p2 - r_p1)*100:+.2f}%",
        'P1_Branch_Spec': f"{s_p1*100:.2f}%",
        'P2_Seq_Spec': f"{s_p2*100:.2f}%",
        'P1_Branch_BalAcc': f"{b_p1*100:.2f}%",
        'P2_Seq_BalAcc': f"{b_p2*100:.2f}%",
        'Delta_BalAcc': f"{(b_p2 - b_p1)*100:+.2f}%",
        'P1_Branch_AUC': a_p1,
        'P2_Seq_AUC': a_p2,
        'Delta_AUC': round(a_p2 - a_p1, 4)
    })

comp_df = pd.DataFrame(comp_rows)
print("=" * 115)
print("     CONSOLIDATED ARCHITECTURE COMPARISON: PIPELINE 1 (BRANCHING) vs PIPELINE 2 (SEQUENTIAL)")
print("=" * 115)
print(comp_df.to_string(index=False))
print("=" * 115 + chr(10))

# Export reports
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)

node_eval_df.to_csv(os.path.join(reports_dir, 'oof_stacking_submodel_nodes_report.csv'), index=False)
report_raw_p1.to_csv(os.path.join(reports_dir, 'oof_stacking_raw_chain_pipeline1_report.csv'), index=False)
report_raw_p2.to_csv(os.path.join(reports_dir, 'oof_stacking_raw_chain_pipeline2_report.csv'), index=False)
report_p1.to_csv(os.path.join(reports_dir, 'oof_stacking_pipeline1_branching_report.csv'), index=False)
report_p2.to_csv(os.path.join(reports_dir, 'oof_stacking_pipeline2_sequential_report.csv'), index=False)
comp_df.to_csv(os.path.join(reports_dir, 'oof_stacking_architecture_comparison.csv'), index=False)
print(f"✓ All comparison and sub-model reports successfully exported to {reports_dir}/")

In [ ]:
# ---------------------------------------------------------
# Step 6: Side-by-Side 5x5 Confusion Matrix Comparison (Pipeline 1 vs Pipeline 2)
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# Compute confusion matrices
cm_p1      = confusion_matrix(y_test, preds_p1, labels=[1, 2, 3, 4, 5])
cm_p1_norm = cm_p1.astype('float') / cm_p1.sum(axis=1)[:, np.newaxis]

cm_p2      = confusion_matrix(y_test, preds_p2, labels=[1, 2, 3, 4, 5])
cm_p2_norm = cm_p2.astype('float') / cm_p2.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

# Left: Pipeline 1 (Branching Tree)
annot_p1 = np.empty_like(cm_p1, dtype=object)
for i in range(5):
    for j in range(5):
        annot_p1[i, j] = f"{cm_p1[i, j]:,}\n({cm_p1_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_p1_norm, annot=annot_p1, fmt='', cmap='Blues', cbar=True, ax=axes[0],
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
axes[0].set_title(
    f"Pipeline 1: Branching Tree Stacking\n"
    f"Macro Balanced Acc: {report_p1.loc[5, 'Balanced_Accuracy']*100:.2f}% | Macro ROC-AUC: {report_p1.loc[5, 'ROC_AUC']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[0].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

# Right: Pipeline 2 (Sequential Peel-Off)
annot_p2 = np.empty_like(cm_p2, dtype=object)
for i in range(5):
    for j in range(5):
        annot_p2[i, j] = f"{cm_p2[i, j]:,}\n({cm_p2_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_p2_norm, annot=annot_p2, fmt='', cmap='Greens', cbar=True, ax=axes[1],
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
axes[1].set_title(
    f"Pipeline 2: Sequential Acuity Peel-off Cascade\n"
    f"Macro Balanced Acc: {report_p2.loc[5, 'Balanced_Accuracy']*100:.2f}% | Macro ROC-AUC: {report_p2.loc[5, 'ROC_AUC']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[1].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[1].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_comp_path = os.path.join(plots_dir, "oof_stacking_architecture_comparison_confusion_matrix.png")
plt.savefig(cm_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'oof_stacking_architecture_comparison_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Side-by-side Confusion Matrix comparison saved to: {cm_comp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Side-by-Side Multiclass ROC-AUC Curves (Pipeline 1 vs Pipeline 2)
# ---------------------------------------------------------
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[1, 2, 3, 4, 5])
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

def plot_roc_curves_on_ax(ax, probs, title_text):
    fpr, tpr, roc_aucs = dict(), dict(), dict()
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs[:, i])
        roc_aucs[i] = auc(fpr[i], tpr[i])
    
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs.ravel())
    roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])
    
    ax.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
    ax.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {roc_aucs['macro']:.4f})", color='#17becf', linestyle='--', linewidth=2.5)
    
    for i in range(5):
        ax.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f})")
    
    ax.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
    ax.set_title(title_text, fontsize=12, fontweight='bold', pad=10)
    ax.legend(loc="lower right", fontsize=9.5, frameon=True, framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_roc_curves_on_ax(axes[0], probs_p1, f"Pipeline 1: Branching Tree ROC Curves (Macro AUC = {report_p1.loc[5, 'ROC_AUC']:.4f})")
plot_roc_curves_on_ax(axes[1], probs_p2, f"Pipeline 2: Sequential Cascade ROC Curves (Macro AUC = {report_p2.loc[5, 'ROC_AUC']:.4f})")

plt.tight_layout()
roc_comp_path = os.path.join(plots_dir, "oof_stacking_architecture_comparison_roc_auc.png")
plt.savefig(roc_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'oof_stacking_architecture_comparison_roc_auc.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Side-by-side ROC Curves saved to: {roc_comp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Export Architecture Comparison Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'pipeline1_branching': {
        'scaler': scaler,
        'l1': p1_l1,
        'l2': p1_l2,
        'l3a': p1_l3a,
        'l3b': p1_l3b,
        'meta': meta_p1,
        'type': 'Branching_Hierarchical_Tree'
    },
    'pipeline2_sequential': {
        'scaler': scaler,
        'l1': p2_l1,
        'l2': p2_l2,
        'l3': p2_l3,
        'l4': p2_l4,
        'meta': meta_p2,
        'type': 'Sequential_Acuity_Peel_Off_Cascade'
    }
}

bundle_file = os.path.join(deploy_dir, 'oof_stacking_architecture_comparison_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    experiment='Hierarchical_Stacking_Architecture_Comparison',
    dataset='datasets/5v_cleandf.RData',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    holdout_test_samples=len(y_test),
    submodel_nodes_metrics=node_eval_df.to_dict(orient='records'),
    raw_chain_p1_metrics=report_raw_p1.to_dict(orient='records'),
    raw_chain_p2_metrics=report_raw_p2.to_dict(orient='records'),
    pipeline1_branching_metrics=report_p1.to_dict(orient='records'),
    pipeline2_sequential_metrics=report_p2.to_dict(orient='records'),
    per_class_comparison=comp_rows
)

manifest_file = os.path.join(deploy_dir, 'oof_stacking_architecture_comparison_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Comparison Bundle   : {bundle_file}")
print(f"✓ Comparison Manifest : {manifest_file}")